# Заказы магазина: от CSV до отчёта по городам

Сколько принесли оплаченные заказы с известной ценой в каждом городе? Вы получите таблицу результатов и отдельный список заказов, цены которых магазин ещё уточняет.

Выполняйте ячейки сверху вниз в Colab или Jupyter с pandas. Для работы в Colab сохраните копию на своём Диске. Учебная выгрузка уже включена в ячейку чтения. При работе с файлом на диске передайте в `pd.read_csv` путь к `orders.csv` с теми же параметрами.

## 1. Прочитайте выгрузку

Какой тип нужен номеру 001? Предскажите, что изменится, если удалить параметр `dtype`. Дату читаем как дату, чтобы позже удобно выбирать дни.

In [ ]:
from io import StringIO
import pandas as pd

csv_text = """order_id,date,city,product,qty,price_kzt,status
001,2026-10-05,Алматы,Тетрадь,2,500,оплачен
002,2026-10-05,Астана,Рюкзак,1,8000,оплачен
003,2026-10-06,Алматы,Ручка,3,200,отменён
004,2026-10-06,Шымкент,Тетрадь,4,500,оплачен
005,2026-10-07,Астана,Ручка,5,200,оплачен
006,2026-10-07,Алматы,Рюкзак,1,уточнить,оплачен
007,2026-10-08,Алматы,Маркер,2,350,оплачен
008,2026-10-08,Шымкент,Маркер,2,,оплачен
009,2026-10-09,Астана,Тетрадь,3,500,возврат
005,2026-10-07,Астана,Ручка,5,200,оплачен
"""
orders = pd.read_csv(
    StringIO(csv_text), dtype={"order_id": "string"}, parse_dates=["date"]
)

In [ ]:
print(orders.head(3).to_string(index=False))
print(orders.shape)
print(orders.dtypes)
assert orders.shape == (10, 7)
assert orders.loc[0, "order_id"] == "001"

В десяти строках может быть меньше десяти заказов. Чтобы проверить это, сравните число строк с количеством разных номеров.

## 2. Найдите повтор и проверьте единицу наблюдения

В этом магазине одна запись соответствует заказу с одним видом товара. Сначала сравните все поля у записей с одинаковым номером.

In [ ]:
same_id = orders["order_id"].duplicated(keep=False)
print(orders.loc[same_id].to_string(index=False))
print("Полных повторов:", orders.duplicated().sum())
unique = orders.drop_duplicates().copy()
assert len(unique) == 9 and unique["order_id"].is_unique

После удаления совпадающей строки номер 005 встречается один раз, и в таблице девять заказов. Если бы под этим номером продавались разные товары, для такой выгрузки пришлось бы пересмотреть правило удаления.

## 3. Оставьте оплаченные заказы и найдите пропуски

Сохраним цену как исходный текст и отдельно получим числовой столбец. Так можно отличить ошибку ввода от пустого поля, когда магазин спросит, какие строки надо перепроверить.

In [ ]:
unique["unit_price"] = pd.to_numeric(unique["price_kzt"], errors="coerce")
paid = unique.loc[unique["status"].eq("оплачен")].copy()
unpriced = paid.loc[paid["unit_price"].isna(), ["order_id", "city", "price_kzt"]]
print(unpriced.to_string(index=False))
assert len(paid) == 7
assert unpriced["order_id"].tolist() == ["006", "008"]

## 4. Посчитайте сумму одного заказа

Строка 001 содержит две тетради по 500 ₸. Найдите её в результате и проверьте, что получилось 1 000 ₸.

In [ ]:
known = paid.loc[paid["unit_price"].notna()].copy()
known["revenue_kzt"] = (known["qty"] * known["unit_price"]).astype("int64")
print(known[["order_id", "city", "revenue_kzt"]].to_string(index=False))
assert known.loc[known["order_id"].eq("001"), "revenue_kzt"].item() == 1000
assert len(known) + len(unpriced) == len(paid)

Все цены учебного магазина заданы целыми тенге. Преобразуем рассчитанную сумму в целое только после проверки пропусков. Если цены содержат дробную часть, сохраните нужную денежную точность.

## 5. Соберите и отсортируйте городской отчёт

Одна строка нового отчёта — один город. Средний чек считаем по тем же заказам, что и выручку.

In [ ]:
report = known.groupby("city", as_index=False).agg(
    orders=("order_id", "nunique"),
    revenue_kzt=("revenue_kzt", "sum"),
)
report["avg_check_kzt"] = report["revenue_kzt"] / report["orders"]
report = report.sort_values("revenue_kzt", ascending=False).reset_index(drop=True)
print(report.to_string(index=False))
assert report["orders"].sum() == known["order_id"].nunique() == 5
assert report["revenue_kzt"].sum() == known["revenue_kzt"].sum() == 12700
assert report.set_index("city").loc["Алматы", "revenue_kzt"] == 1700

## 6. Сохраните проверяемый результат

`index=False` исключает служебную нумерацию строк из CSV. Запишем учебный отчёт в `city_report_lesson.csv` и заново прочитаем его.

In [ ]:
report.to_csv("city_report_lesson.csv", index=False)
restored = pd.read_csv("city_report_lesson.csv")
pd.testing.assert_frame_equal(restored, report, check_dtype=False)
print("Проверено:", len(restored), "города,", restored["revenue_kzt"].sum(), "₸")

## Объясните вывод своими словами

Почему в итоговую таблицу вошли пять заказов из десяти строк выгрузки? Сколько из оставшихся семи оплаченных заказов ждут уточнения? Как изменился бы знаменатель среднего чека, если бы неизвестную цену записали нулём?